CELL 1
# FictionalCart - Returns Dataset Generation

## Notebook 04 - Returns_Generation.ipynb

### Objective

Generate a realistic Returns dataset for the FictionalCart e-commerce business using the master Orders dataset.

The Returns dataset will:

- Include only orders with `Order_Status = "Returned"`
- Preserve referential integrity with Orders, Customers, and Products
- Generate realistic return dates and return reasons
- Calculate valid refund amounts
- Validate all business rules before export

### Input Files

- Customer_Master.csv
- Product_Master.csv
- Orders.csv

### Output File

- Returns.csv

### Development Workflow

1. Imports
2. Configuration
3. Load Master Data
4. Helper Functions
5. Returns Generation
6. Validation
7. Export
8. Freeze Summary

In [1]:
#CELL 2 - IMPORT LIBRARIES

import pandas as pd
import numpy as np

In [2]:
#CELL 3 - CONFIGURATION

# Random Seed (for reproducibility)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# File Paths
CUSTOMER_FILE = "Customer_Master.csv"
PRODUCT_FILE = "Product_Master.csv"
ORDERS_FILE = "Orders.csv"
OUTPUT_FILE = "Returns.csv"

# Return Business Rules
MIN_RETURN_DAYS = 2
MAX_RETURN_DAYS = 30

In [24]:
#CELL 4 - LOAD MASTER DATA

customers_df = pd.read_csv(CUSTOMER_FILE)
products_df = pd.read_csv(PRODUCT_FILE)
orders_df = pd.read_csv(ORDERS_FILE)

In [4]:
#CELL 5 - CONVERT DATE COLUMNS

customers_df["Join_Date"] = pd.to_datetime(customers_df["Join_Date"])

products_df["Launch_Date"] = pd.to_datetime(products_df["Launch_Date"])

orders_df["Order_Date"] = pd.to_datetime(orders_df["Order_Date"])

In [5]:
#CELL 6 - INITIAL DATASET VALIDATION

print("=" * 60)
print("MASTER DATA VALIDATION")
print("=" * 60)

print(f"Customers : {len(customers_df)} rows")
print(f"Products  : {len(products_df)} rows")
print(f"Orders    : {len(orders_df)} rows")

print("\nMissing Values")
print("-" * 60)

print(f"Customers : {customers_df.isna().sum().sum()}")
print(f"Products  : {products_df.isna().sum().sum()}")
print(f"Orders    : {orders_df.isna().sum().sum()}")

MASTER DATA VALIDATION
Customers : 750 rows
Products  : 200 rows
Orders    : 5000 rows

Missing Values
------------------------------------------------------------
Customers : 0
Products  : 0
Orders    : 0


In [6]:
#CELL 7 - CREATE BASE RETURNS DATASET

returns_df = (
    orders_df.loc[orders_df["Order_Status"] == "Returned"]
    .copy()
    .reset_index(drop=True)
)

print("=" * 60)
print("BASE RETURNS DATASET")
print("=" * 60)
print(f"Returned Orders : {len(returns_df)}")

BASE RETURNS DATASET
Returned Orders : 253


In [7]:
#CELL 8 - VALIDATE BASE RETURNS DATASET

print("\nOrder Status Distribution")
print("-" * 60)

print(returns_df["Order_Status"].value_counts())

print("\nReturned Orders Validation")

if (returns_df["Order_Status"] == "Returned").all():
    print("✅ All records are Returned orders.")
else:
    print("❌ Invalid order statuses detected.")


Order Status Distribution
------------------------------------------------------------
Order_Status
Returned    253
Name: count, dtype: int64

Returned Orders Validation
✅ All records are Returned orders.


In [8]:
#CELL 9 - GENERATE RETURN_ID

returns_df.insert(
    0,
    "Return_ID",
    [f"R{i:05d}" for i in range(1, len(returns_df) + 1)]
)

print(returns_df[["Return_ID", "Order_ID"]].head())

  Return_ID Order_ID
0    R00001  O000002
1    R00002  O000003
2    R00003  O000076
3    R00004  O000078
4    R00005  O000122


In [9]:
#CELL 10 - VALIDATE RETURN_ID

print("=" * 60)
print("RETURN ID VALIDATION")
print("=" * 60)

print(f"Total Returns      : {len(returns_df)}")
print(f"Unique Return_IDs  : {returns_df['Return_ID'].nunique()}")

if returns_df["Return_ID"].is_unique:
    print("✅ Return_ID values are unique.")
else:
    print("❌ Duplicate Return_ID values found.")

RETURN ID VALIDATION
Total Returns      : 253
Unique Return_IDs  : 253
✅ Return_ID values are unique.


In [10]:
#CELL 11 - HELPER FUNCTION FOR RETURN DATE

def generate_return_date(order_date):
    """
    Generate a realistic return request date.

    Business Rule:
    Return request must occur between
    2 and 30 days after the order date.
    """

    days_after_order = np.random.randint(
        MIN_RETURN_DAYS,
        MAX_RETURN_DAYS + 1
    )

    return order_date + pd.Timedelta(days=days_after_order)

In [11]:
#CELL 12 - GENERATE RETURN_DATE

returns_df["Return_Date"] = (
    returns_df["Order_Date"]
    .apply(generate_return_date)
)

In [12]:
#CELL 13 - VALIDATE RETURN_DATE

days_difference = (
    returns_df["Return_Date"]
    - returns_df["Order_Date"]
).dt.days

print("=" * 60)
print("RETURN DATE VALIDATION")
print("=" * 60)

print(f"Earliest Return Gap : {days_difference.min()} days")
print(f"Latest Return Gap   : {days_difference.max()} days")

if (
    days_difference.between(
        MIN_RETURN_DAYS,
        MAX_RETURN_DAYS
    ).all()
):
    print("✅ Return_Date validation passed.")
else:
    print("❌ Invalid Return_Date detected.")

RETURN DATE VALIDATION
Earliest Return Gap : 2 days
Latest Return Gap   : 30 days
✅ Return_Date validation passed.


In [13]:
#CELL 14 - FREEZE HELPER FUNCTION

print("✅ Helper Function Frozen:")
print("- generate_return_date()")

✅ Helper Function Frozen:
- generate_return_date()


In [14]:
#CELL 15 - DEFINE RETURN REASON DISTRIBUTION

RETURN_REASONS = [
    "Damaged Product",
    "Wrong Item Received",
    "Defective Product",
    "Changed Mind",
    "Size/Fit Issue",
    "Late Delivery",
    "Other"
]

RETURN_REASON_PROBABILITIES = [
    0.30,
    0.20,
    0.18,
    0.15,
    0.10,
    0.05,
    0.02
]

In [15]:
#CELL 16 - VELIDATE PROBBAILITY DISTRIBUTION

total_probability = sum(RETURN_REASON_PROBABILITIES)

print("=" * 60)
print("RETURN REASON PROBABILITY VALIDATION")
print("=" * 60)

print(f"Total Probability : {total_probability:.2f}")

if np.isclose(total_probability, 1.0):
    print("✅ Probability validation passed.")
else:
    print("❌ Probability validation failed.")

RETURN REASON PROBABILITY VALIDATION
Total Probability : 1.00
✅ Probability validation passed.


In [16]:
#CELL 17 - GENERATE RETURN_REASON

returns_df["Return_Reason"] = np.random.choice(
    RETURN_REASONS,
    size=len(returns_df),
    p=RETURN_REASON_PROBABILITIES
)

In [17]:
#CELL 18 - VALIDATE RETURN REASON DISTRIBUTION

print("=" * 60)
print("RETURN REASON DISTRIBUTION")
print("=" * 60)

reason_distribution = (
    returns_df["Return_Reason"]
    .value_counts()
    .sort_values(ascending=False)
)

print(reason_distribution)

RETURN REASON DISTRIBUTION
Return_Reason
Damaged Product        72
Wrong Item Received    54
Defective Product      47
Changed Mind           33
Size/Fit Issue         20
Late Delivery          19
Other                   8
Name: count, dtype: int64


In [18]:
#CELL 19 - GENERATE REFUND AMOUNT

returns_df["Refund_Amount"] = returns_df["Order_Amount"]

In [19]:
#CELL 20 - VALIDATE REFUND AMOUNT

print("=" * 60)
print("REFUND AMOUNT VALIDATION")
print("=" * 60)

invalid_refunds = (
    returns_df["Refund_Amount"]
    > returns_df["Order_Amount"]
).sum()

print(f"Invalid Refund Amounts : {invalid_refunds}")

if invalid_refunds == 0:
    print("✅ Refund Amount validation passed.")
else:
    print("❌ Refund Amount validation failed.")

REFUND AMOUNT VALIDATION
Invalid Refund Amounts : 0
✅ Refund Amount validation passed.


In [20]:
#CELL 21 - CREATE THE FINAL RETURNS DATASET

returns_df = returns_df[
    [
        "Return_ID",
        "Order_ID",
        "Customer_ID",
        "Product_ID",
        "Return_Date",
        "Return_Reason",
        "Refund_Amount"
    ]
].copy()

In [21]:
#CELL 22 - FINAL DATASET VALIDATION

print("=" * 60)
print("FINAL RETURNS DATASET VALIDATION")
print("=" * 60)

print(f"Rows    : {len(returns_df)}")
print(f"Columns : {returns_df.shape[1]}")

print("\nColumn Names")
print("-" * 60)

print(list(returns_df.columns))

print("\nMissing Values")
print("-" * 60)

print(returns_df.isnull().sum())

print("\nDuplicate Return_IDs")
print("-" * 60)

print(returns_df["Return_ID"].duplicated().sum())

print("\nDuplicate Order_IDs")
print("-" * 60)

print(returns_df["Order_ID"].duplicated().sum())

FINAL RETURNS DATASET VALIDATION
Rows    : 253
Columns : 7

Column Names
------------------------------------------------------------
['Return_ID', 'Order_ID', 'Customer_ID', 'Product_ID', 'Return_Date', 'Return_Reason', 'Refund_Amount']

Missing Values
------------------------------------------------------------
Return_ID        0
Order_ID         0
Customer_ID      0
Product_ID       0
Return_Date      0
Return_Reason    0
Refund_Amount    0
dtype: int64

Duplicate Return_IDs
------------------------------------------------------------
0

Duplicate Order_IDs
------------------------------------------------------------
0


In [22]:
#CELL 23 - EXPORT RETURNS DATASET

returns_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"File Saved : {OUTPUT_FILE}")
print(f"Rows       : {len(returns_df)}")
print(f"Columns    : {returns_df.shape[1]}")

EXPORT COMPLETE
File Saved : Returns.csv
Rows       : 253
Columns    : 7


In [23]:
#CELL 24 - FREEZE SUMMARY

print("=" * 60)
print("NOTEBOOK 04 COMPLETED")
print("=" * 60)

print("Dataset Generated : Returns.csv")
print(f"Rows              : {len(returns_df)}")
print(f"Columns           : {returns_df.shape[1]}")

print("\nNotebook Status")
print("-" * 60)

print("✓ Business Rules Frozen")
print("✓ Dataset Schema Frozen")
print("✓ Helper Functions Frozen")
print("✓ Validation Passed")
print("✓ Returns.csv Exported")

print("\nReady for the next notebook.")

NOTEBOOK 04 COMPLETED
Dataset Generated : Returns.csv
Rows              : 253
Columns           : 7

Notebook Status
------------------------------------------------------------
✓ Business Rules Frozen
✓ Dataset Schema Frozen
✓ Helper Functions Frozen
✓ Validation Passed
✓ Returns.csv Exported

Ready for the next notebook.
